# ETL Notebook (Simple)

Reads a CSV from a raw path, filters out rows where `value` is null, and writes a Delta output under a processed path organized by `run_date`.

This version avoids widgets/dbutils and resolves parameters from a control table when available.

In [ ]:
from datetime import date
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.getOrCreate()


In [ ]:
# Basic configuration + parameter resolution (control table > defaults)\n
DEFAULT_ENV = 'dev'
DEFAULT_RAW_BASE_PATH = '/Volumes/workspace/default/raw'
DEFAULT_PROCESSED_BASE_PATH = '/Volumes/workspace/default/processed'
DEFAULT_INPUT_FILENAME = 'input.csv'

def get_param_from_table(spark, env, key, default=None):
    try:
        df = spark.read.table('workspace.default.control_parameters')
        row = df.filter((df.env == env) & (df.key == key)).select('value').first()
        if row and row.value is not None:
            return row.value
    except Exception:
        pass
    return default

# Resolve env (use default or hard-coded selection)
env = DEFAULT_ENV

# Resolve run_date (table -> today)
run_date = get_param_from_table(spark, env, 'run_date', default=date.today().strftime('%Y-%m-%d'))

# Resolve base paths (table -> defaults)
raw_base_path = get_param_from_table(spark, env, 'raw_base_path', default=DEFAULT_RAW_BASE_PATH)
processed_base_path = get_param_from_table(spark, env, 'processed_base_path', default=DEFAULT_PROCESSED_BASE_PATH)

# Resolve input filename (table -> default)
input_filename = get_param_from_table(spark, env, 'input_filename', default=DEFAULT_INPUT_FILENAME)

print(f'env={env}, run_date={run_date}')
print(f'raw_base_path={raw_base_path}, processed_base_path={processed_base_path}, input={input_filename}')


In [ ]:
# ETL
input_path = f"{raw_base_path}/{input_filename}"
output_path = f"{processed_base_path}/{run_date}/Notebook"

df = spark.read.option('header', True).csv(input_path)

df_filtered = df.filter(col('value').isNotNull())

print(f'Read {df.count()} rows; writing {df_filtered.count()} non-null rows to {output_path}')

df_filtered.write.format('delta').mode('overwrite').save(output_path)
